# Pamoka 10 - DI agentai produkcijoje

Šioje pamokoje sužinosite apie **produkcinio lygio sprendimus** DI agentams, naudojant Microsoft Agent Framework (Python). Aptarsime:

- **Stebėjimas** — pridėti laiko matavimą ir žurnalavimą agentų sąveikoms
- **Įvertinimas** — naudojant vertinimo agentą atsakymų kokybei įvertinti
- **Kaštų valdymas** — strategijos tokenų optimizavimui ir modelio pasirinkimui

Scenarijus — **kelionių agentas**, padedantis vartotojams planuoti keliones, su pridėta stebėsena ir vertinimu.


## Nustatymas


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Gamybos svarstymai

Perkelti dirbtinio intelekto agentus iš prototipų į gamybą reikalauja atidaus dėmesio trims pagrindinėms sritims:

1. **Stebėjimas** — Jums reikia matomumo, ką agentas daro, kiek tai užtrunka ir kokius įrankius jis kviečia. Be sekimo ir žurnalo įrašų (tracing ir logging) gamybinių problemų derinimas yra beveik neįmanomas.

2. **Vertinimas** — Automatiniai kokybės patikrinimai užtikrina, kad agento atsakymai laikui bėgant išliktų tikslūs, pilni ir naudingi. Vertinimo agentas gali įvertinti atsakymus pagal apibrėžtus kriterijus.

3. **Išlaidų valdymas** — Tokenų naudojimas tiesiogiai įtakoja išlaidas. Tokios strategijos kaip promptų optimizavimas, modelio pasirinkimas ir talpyklos naudojimas (caching) padeda kontroliuoti išlaidas neaukojant kokybės.


## Kuriame stebimą agentą

Mes apibrėžiame kelionės įrankius ir apvyniojame agento kvietimą laiko matuokliu, kad galėtume stebėti vėlavimą. Gamybinėje aplinkoje integruotumėte OpenTelemetry arba panašų sekimo backend'ą.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Vertinimo modeliai

Bendras gamybos modelis yra naudoti antrą agentą kaip **vertintoją**. Vertintojas įvertina pagrindinio agento atsakymą pagal iš anksto nustatytus kriterijus, tokius kaip išsamumas, tikslumas ir naudingumas.

Tai leidžia:
- Automatizuotos kokybės patikros prieš atsakymus pasiekiant vartotojus
- Regresijos aptikimas, kai keičiasi užklausos arba modeliai
- Nuolatinis agento našumo stebėjimas laikui bėgant


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Išlaidų valdymo strategijos

Išlaidų kontrolė yra kritiškai svarbi gamybiniams AI agentams. Čia pateiktos pagrindinės strategijos:

| Strategija | Aprašymas |
|---|---|
| **Užklausos optimizavimas** | Išlaikykite sistemos nurodymus glaustus. Pašalinkite perteklinį kontekstą, kad sumažintumėte įvesties tokenų skaičių. |
| **Modelio pasirinkimas** | Naudokite mažesnius, pigesnius modelius (pvz. GPT-4o-mini) paprastoms užduotims, tokioms kaip klasifikacija ar ištraukimas, o didesnius modelius palikite sudėtingam samprotavimui. |
| **Kešavimas** | Kešuokite įrankių rezultatus ir dažnas užklausas, kad išvengtumėte perteklinių API užklausų. |
| **Tokenų biudžetai** | Nustatykite `max_tokens` ribas, kad išvengtumėte netikėtai ilgų atsakymų. |
| **Grupavimas** | Sujunkite kelias vartotojo užklausas į vieną API užklausą, kai tai įmanoma. |

Praktikoje gerai veikia sluoksniuota prieiga: paprastas užklausas nukreipkite į greitą, nebrangų modelį, o sudėtingas užklausas nukreipkite tik į galingesnį modelį.


## Santrauka

Šiame pamokoje išmokote, kaip:

1. **Pridėti stebimumą** prie agentų sąveikų naudojant laiko matavimą ir žurnalavimą, taip sudarant pagrindą sekimui ir stebėsenai.
2. **Automatiškai įvertinti agentų atsakymus** naudojant vertinimo agentą, kuris įvertina išsamumą, tikslumą ir naudingumą.
3. **Valdyti kaštus** per užklausų optimizavimą, modelių pasirinkimą, talpyklavimą ir tokenų biudžetus.

Šios gamybinės praktikos padeda užtikrinti, kad jūsų dirbtinio intelekto agentai būtų patikimi, matuojami ir ekonomiški didelėse apimtyse.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Atsakomybės apribojimas:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors stengiamės užtikrinti tikslumą, atkreipkite dėmesį, kad automatizuoti vertimai gali turėti klaidų ar netikslumų. Originalų dokumentą jo gimtąja kalba reikėtų laikyti autoritetingu šaltiniu. Jei informacija yra kritiškai svarbi, rekomenduojamas profesionalus žmogaus vertimas. Mes neprisiimame atsakomybės už bet kokius nesusipratimus ar neteisingą aiškinimą, kilusius naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
